# Session 1: a neuron adds up signals and fires
**A. Real-world problem.** Your nerve cells talk with brief electrical pulses called
*spikes*. A neuron collects signals from other cells on its branches (dendrites), adds
them up in its cell body, and, if the total voltage crosses a threshold, fires one spike
that runs down its long wire (the axon) to the next cells. A weak input does nothing; a
strong input does not make a bigger spike, only more of them. Today you build the simplest
model of that idea and draw its voltage-versus-time trace.

**Plan.** (1) a voltage that rises with input and leaks away, (2) a threshold that fires a
fixed-size spike, (3) how long the spike takes to travel down an axon.

*General education only. This is a simplified model, not medical information.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

V_REST = -70.0     # resting voltage, mV (a typical textbook value)
V_THRESH = -55.0   # threshold voltage, mV (typical)
V_SPIKE = 40.0     # the drawn height of a spike, mV (every spike has the same height)
LEAK = 0.1         # fraction of the distance back to rest that leaks away each millisecond
REFRACTORY = 2     # milliseconds after a spike during which the neuron cannot fire again

## Step 1: a voltage that rises with input and leaks away
Time moves in steps of 1 ms. Each step the input pushes the voltage up by `drive` (mV),
and the leak pulls it back towards rest:

    V_new = V + drive - LEAK * (V - V_rest)

If the drive is constant the voltage climbs towards V_rest + drive / LEAK and stops there.
That final level must be above the threshold for the neuron ever to fire, so the smallest
drive that can fire is  drive_min = LEAK * (V_thresh - V_rest) = 0.1 * 15 = 1.5 mV per ms.

In [ ]:
def final_level(drive):
    """Voltage (mV) the neuron would settle at if it never fired."""
    return V_REST + drive / LEAK


def smallest_drive():
    """Smallest constant drive (mV per ms) that can ever reach the threshold."""
    return LEAK * (V_THRESH - V_REST)


def steps_to_first_spike(drive):
    """Hand calculation: number of 1 ms steps until V reaches the threshold (None if never)."""
    if final_level(drive) <= V_THRESH:
        return None
    need = (V_THRESH - V_REST) * LEAK / drive          # (1 - LEAK)^n must fall to 1 - need
    return int(np.ceil(np.log(1.0 - need) / np.log(1.0 - LEAK) - 1e-9))

## Step 2: the threshold and the spike
When V reaches the threshold the neuron fires: we draw a spike of fixed height, reset V to
rest, and block firing for a short refractory time. The spike height never depends on how
strong the input was: *all or nothing*.

In [ ]:
def toy_neuron(drive, steps=100):
    """Simulate `steps` milliseconds. Returns (voltage trace in mV, list of spike times in ms).
    `drive` is a number (constant) or an array with one value per step."""
    drive = np.broadcast_to(np.asarray(drive, dtype=float), (steps,))
    v = V_REST
    wait = 0
    trace = []
    spikes = []
    for n in range(steps):
        if wait > 0:
            wait -= 1
            v = V_REST
            trace.append(V_REST)
            continue
        v = v + drive[n] - LEAK * (v - V_REST)
        if v >= V_THRESH:
            spikes.append(n + 1)
            trace.append(V_SPIKE)
            v = V_REST
            wait = REFRACTORY
        else:
            trace.append(v)
    return np.array(trace), spikes

## Step 3: the spike travels down the axon
A spike moves along the axon at a roughly constant speed, so the delay is
`delay = length / speed`. Typical speeds range from about 1 m/s in thin unmyelinated fibres
to over 100 m/s in thick myelinated ones.

In [ ]:
def travel_time_ms(length_m, speed_m_per_s):
    return 1000.0 * length_m / speed_m_per_s

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle

    OUT.mkdir(exist_ok=True)
    print("smallest drive that can fire:", smallest_drive(), "mV per ms")
    for d in (1.0, 1.5, 2.0, 3.0):
        tr, sp = toy_neuron(d)
        print(f"drive {d}: {len(sp)} spikes in 100 ms, first at {sp[0] if sp else None} ms,"
              f" hand calculation {steps_to_first_spike(d)}, highest value {tr.max():.1f} mV")

    # picture of a neuron
    fig, ax = plt.subplots(figsize=(8, 3))
    for ang, ln in ((140, 1.6), (180, 1.8), (220, 1.6)):
        a = np.radians(ang)
        ax.plot([0, ln * np.cos(a)], [0, ln * np.sin(a)], color="#2a9d8f", lw=3)
    ax.add_patch(Circle((0, 0), 0.5, color="#f4a261"))
    ax.plot([0.5, 9.5], [0, 0], color="#264653", lw=2.5)
    ax.annotate("dendrites\n(collect signals)", (-1.6, 0.9), ha="center")
    ax.annotate("cell body\n(adds them up)", (0, 0.65), ha="center")
    ax.annotate("axon (carries the spike)", (5, 0.25), ha="center")
    ax.annotate("to next cells", (9.5, -0.5), ha="center")
    ax.set_xlim(-3, 10.5)
    ax.set_ylim(-1.4, 1.6)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "neuron_picture.png", dpi=120)

    # voltage traces
    fig2, axs = plt.subplots(3, 1, figsize=(7, 6.5), sharex=True)
    for ax2, d in zip(axs, (1.0, 2.0, 4.0)):
        tr, sp = toy_neuron(d)
        ax2.plot(np.arange(1, 101), tr)
        ax2.axhline(V_THRESH, color="red", ls="--", lw=1)
        ax2.set_ylabel("voltage (mV)")
        ax2.set_title(f"drive {d} mV per ms: {len(sp)} spikes", fontsize=9)
        ax2.grid(alpha=0.3)
    axs[0].text(60, V_THRESH + 3, "threshold", color="red", fontsize=8)
    axs[-1].set_xlabel("time (ms)")
    fig2.tight_layout()
    fig2.savefig(OUT / "voltage_traces.png", dpi=120)
    print("delay for 0.5 m at 50 m/s:", travel_time_ms(0.5, 50.0), "ms")

## Coding tasks
1. **Easy.** In `lesson.py` change the leak to 0.2 and predict the smallest drive that can fire before you run it (use the formula d = k (V_th - V_rest)).
2. **Stretch.** Write `first_spike_step(drive)` that finds the first spike by looping over `toy_neuron`, and compare it with `steps_to_first_spike` for drives 2.0, 3.0 and 4.0.
3. **Explain.** Explain why doubling the drive does not exactly double the number of spikes in 100 ms. (Hint: look at the time it takes the voltage to climb and at the refractory time.)

## Assessment
- **A. Conceptual.** A stronger stimulus does not make a bigger spike. So how does a nerve tell the brain that a touch is stronger?
- **B. Numerical.** For a constant drive of 3.0 mV per ms find the level the voltage would settle at, and the step at which the first spike happens (hand calculation, then check with the code).
- **C. Graph and simulation interpretation.** In the trace for drive 1.0 mV per ms the voltage rises and then flattens out below the threshold line. What level does it flatten at, and why is there no spike?
- **D. Design and optimisation challenge.** Design: a model neuron must fire at least 10 times in 100 ms, and a spike must cross a 0.5 m axon in under 5 ms. Choose a drive (in steps of 0.5 mV per ms) and an axon speed, and say what trade-off you see.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("the smallest drive that can fire is 1.5 mV per ms (hand: 0.1 x 15)", abs(L.smallest_drive() - 1.5) < 1e-12))
    tr, sp = L.toy_neuron(1.0)
    checks.append(("weak input (1.0 mV per ms) never fires and stays below threshold", len(sp) == 0 and tr.max() < L.V_THRESH))
    checks.append(("final level for drive 1.0 is -60 mV", abs(L.final_level(1.0) - (-60.0)) < 1e-12))
    tr15, sp15 = L.toy_neuron(1.5)
    checks.append(("edge case: drive exactly 1.5 only approaches the threshold and never fires", len(sp15) == 0))
    tr2, sp2 = L.toy_neuron(2.0)
    checks.append((f"worked example: drive 2.0 first fires at step 14 (hand: ceil(ln 0.25 / ln 0.9) = 14), got {sp2[0] if sp2 else None}",
                   L.steps_to_first_spike(2.0) == 14 and sp2 and sp2[0] == 14))
    checks.append(("worked example: drive 2.0 gives 6 spikes in 100 ms", len(sp2) == 6))
    tr3, sp3 = L.toy_neuron(3.0)
    tr4, sp4 = L.toy_neuron(4.0)
    checks.append(("worked example: drive 3.0 gives 11 spikes and drive 4.0 gives 14 spikes", len(sp3) == 11 and len(sp4) == 14))
    checks.append(("all or nothing: every spike has the same height whatever the drive", tr2.max() == tr3.max() == tr4.max() == L.V_SPIKE))
    checks.append(("a stronger drive fires more often", len(sp4) > len(sp3) > len(sp2) > 0))
    gaps = np.diff(sp2)
    checks.append(("refractory period: spikes are at least 3 ms apart even for a huge drive", bool(np.all(np.diff(L.toy_neuron(50.0)[1]) >= 3))))
    checks.append(("drive 2.0: spikes are regularly spaced (gaps all 16 or 17 ms)", bool(np.all((gaps >= 16) & (gaps <= 17)))))
    checks.append(("without any input the neuron sits at rest", np.allclose(L.toy_neuron(0.0)[0], L.V_REST)))
    pulse = np.zeros(100)
    pulse[10] = 20.0
    checks.append(("one big pulse of 20 mV (above 15 mV) fires at once", L.toy_neuron(pulse)[1] == [11]))
    checks.append(("one small pulse of 10 mV does not fire", L.toy_neuron(np.where(np.arange(100) == 10, 10.0, 0.0))[1] == []))
    checks.append(("axon delay: 0.5 m at 50 m/s takes 10 ms", abs(L.travel_time_ms(0.5, 50.0) - 10.0) < 1e-12))
    checks.append(("axon delay: 1 m at 100 m/s takes 10 ms and 1 m at 1 m/s takes 1000 ms", abs(L.travel_time_ms(1.0, 100.0) - 10.0) < 1e-12 and abs(L.travel_time_ms(1.0, 1.0) - 1000.0) < 1e-9))
    checks.append(("worked example: hand calculation ln 0.25 / ln 0.9 = 13.16, so 13 steps are not enough (V = -55.08 mV) and 14 are (V = -54.58 mV)",
                   abs(np.log(0.25) / np.log(0.9) - 13.158) < 0.005 and abs(-70 + 20 * (1 - 0.9 ** 13) + 55.08) < 0.005 and abs(-70 + 20 * (1 - 0.9 ** 14) + 54.58) < 0.005))
    checks.append(("question B: drive 3.0 settles at -40 mV and first fires at step 7 (hand: ln 0.5 / ln 0.9 = 6.58)",
                   abs(L.final_level(3.0) + 40.0) < 1e-12 and L.steps_to_first_spike(3.0) == 7 == sp3[0] and abs(np.log(0.5) / np.log(0.9) - 6.579) < 0.005))
    checks.append(("question D: drive 2.5 gives 9 spikes and drive 3.5 gives 12 spikes in 100 ms", len(L.toy_neuron(2.5)[1]) == 9 and len(L.toy_neuron(3.5)[1]) == 12))
    checks.append(("question D: a 0.5 m axon needs more than 100 m/s to deliver in under 5 ms (exactly 5 ms at 100 m/s)", abs(L.travel_time_ms(0.5, 100.0) - 5.0) < 1e-12))
    checks.append(("misconception: with an enormous drive the neuron fires every 3 ms (34 spikes in 100 ms), never faster", bool(np.all(np.diff(L.toy_neuron(50.0)[1]) == 3)) and len(L.toy_neuron(50.0)[1]) == 34))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
